<a href="https://colab.research.google.com/github/hguan001/git-java-lab/blob/main/ProyectoMineria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Bloque 1**

In [ ]:
!pip install -q datasets gensim sentence-transformers pysentimiento plotly nltk scikit-learn joblib tqdm

import os
import nltk

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
for folder in ["data/raw", "data/processed", "data/embeddings/models", "src", "results/plots", "results/tables"]:
    os.makedirs(folder, exist_ok=True)

print("¡Entorno listo y carpetas creadas!")

¡Entorno listo y carpetas creadas!


En el primer bloque se descargan las librerias necesarias y construye la estructura de directorios requerida para organizar el código.

# **Bloque 2**

In [ ]:
%%writefile src/r1_data_prep.py
import os
import re
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from datasets import load_dataset
from sklearn.model_selection import train_test_split

nltk.download('stopwords', quiet=True)
STOPWORDS = set(stopwords.words('english'))

def load_pubmed_safe(n_samples: int = 150) -> pd.DataFrame:
    print(f"Conectando a PubMed (formato seguro) para extraer {n_samples} abstracts...")
    ds = load_dataset("ccdv/pubmed-summarization", split="train", streaming=True)
    data = [{"text": row["article"]} for row in ds.take(n_samples)]
    df = pd.DataFrame(data)
    os.makedirs("data/raw", exist_ok=True)
    df.to_csv("data/raw/medal_train_sample.csv", index=False)
    return df

def clean_biomedical_text(text: str) -> str:
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return ' '.join(tokens)

def run_preprocessing_and_eda(test_size: float = 0.20, seed: int = 42):
    df = load_pubmed_safe(n_samples=150)
    text_col = 'text'

    n_initial = len(df)
    df = df.dropna(subset=[text_col]).drop_duplicates(subset=[text_col]).reset_index(drop=True)

    df['raw_text'] = df[text_col].astype(str)
    df['clean_text'] = df['raw_text'].apply(clean_biomedical_text)
    df = df[df['clean_text'].str.strip().astype(bool)].reset_index(drop=True)

    train_df, test_df = train_test_split(df, test_size=test_size, random_state=seed)

    print(f"\n--- RESUMEN ---")
    print(f"Descarga y limpieza exitosa. Train: {len(train_df)} | Test: {len(test_df)}")

    os.makedirs("data/processed", exist_ok=True)
    train_df.to_csv("data/processed/train_clean.csv", index=False)
    test_df.to_csv("data/processed/test_clean.csv", index=False)
    return train_df, test_df

if __name__ == "__main__":
    run_preprocessing_and_eda()

Overwriting src/r1_data_prep.py


Descarga los datos medicos del dataset Zenodo-PubMed abstracts (Medal), elimina los enlaces y signo de puntuacion manteniendo la terminología clínica, separa los datos en Entrenamiento y Test, y cuenta las palabras para la memoria.

# **Bloque 3**

In [ ]:
%%writefile src/r1_vectorizers.py
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from gensim.models import Word2Vec
from sentence_transformers import SentenceTransformer
from pysentimiento import create_analyzer

def save_matrix(name: str, X_tr: np.ndarray, X_te: np.ndarray):
    X_tr, X_te = np.asarray(X_tr, dtype=np.float32), np.asarray(X_te, dtype=np.float32)
    np.save(f"data/embeddings/{name}_train.npy", X_tr)
    np.save(f"data/embeddings/{name}_test.npy", X_te)
    print(f"[{name.upper()}] guardado. Forma Train: {X_tr.shape}")

def generate_all_representations():
    train_df = pd.read_csv("data/processed/train_clean.csv")
    test_df = pd.read_csv("data/processed/test_clean.csv")
    tr_cln, te_cln = train_df['clean_text'].fillna("").tolist(), test_df['clean_text'].fillna("").tolist()
    tr_raw, te_raw = train_df['raw_text'].fillna("").tolist(), test_df['raw_text'].fillna("").tolist()

    # BoW y TF-IDF
    bow = CountVectorizer(max_features=2500)
    save_matrix("bow", bow.fit_transform(tr_cln).toarray(), bow.transform(te_cln).toarray())
    joblib.dump(bow, "data/embeddings/models/bow.pkl")

    tfidf = TfidfVectorizer(max_features=2500)
    save_matrix("tfidf", tfidf.fit_transform(tr_cln).toarray(), tfidf.transform(te_cln).toarray())
    joblib.dump(tfidf, "data/embeddings/models/tfidf.pkl")

    # LSA
    lsa = TruncatedSVD(n_components=100, random_state=42)
    save_matrix("lsa", lsa.fit_transform(tfidf.transform(tr_cln)), lsa.transform(tfidf.transform(te_cln)))

    # Sentence-Transformers
    print("Calculando SBERT (Contextual)...")
    sbert = SentenceTransformer("all-MiniLM-L6-v2")
    save_matrix("sbert", sbert.encode(tr_raw), sbert.encode(te_raw))

    # Emoción (Pysentimiento)
    print("Calculando Emociones...")
    analyzer = create_analyzer(task="emotion", lang="en")
    emo_labels = ['others', 'joy', 'sadness', 'anger', 'surprise', 'disgust', 'fear']
    def get_emo(texts):
        preds = analyzer.predict([t[:400] for t in texts])
        return np.array([[p.probas[k] for k in emo_labels] for p in preds])
    save_matrix("emotion", get_emo(tr_raw), get_emo(te_raw))

if __name__ == "__main__":
    generate_all_representations()

Overwriting src/r1_vectorizers.py


Extrae las características de los textos creando las matrices numéricas .npy. Utiliza tanto aproximaciones dispersas (BoW, TF-IDF) como densas (Transformers), lo cual es el paso previo fundamental para poder calcular la medida de similitud/disimilitud entre instancias.

# **Bloque 4**

In [ ]:
%%writefile src/r1_visualization.py
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import TfidfVectorizer

def extract_top_terms(clean_texts, labels, top_n=4):
    vec = TfidfVectorizer(max_features=1000)
    X_tfidf = vec.fit_transform(clean_texts).toarray()
    vocab = np.array(vec.get_feature_names_out())

    cluster_desc = {}
    for c in np.unique(labels):
        if c == -1: cluster_desc[c] = "Ruido"
        else:
            mean_w = X_tfidf[labels == c].mean(axis=0)
            cluster_desc[c] = f"Clúster {c}: {', '.join(vocab[mean_w.argsort()[::-1][:top_n]])}"
    return cluster_desc

def plot_interactive(X, labels, raw_texts, clean_texts, rep_name="sbert"):
    c_map = extract_top_terms(clean_texts, labels)
    named_labels = [c_map[c] for c in labels]
    snippets = [t[:150] + "..." for t in raw_texts]

    # t-SNE (Proyección no lineal)
    X_pre = PCA(n_components=50).fit_transform(X) if X.shape[1] > 50 else X
    coords = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_pre)

    df_viz = pd.DataFrame({"X": coords[:, 0], "Y": coords[:, 1], "Cluster": named_labels, "Texto": snippets})
    fig = px.scatter(df_viz, x="X", y="Y", color="Cluster", hover_data=["Cluster", "Texto"],
                     title=f"t-SNE 2D - Representación {rep_name.upper()}")

    out = f"results/plots/{rep_name}_tsne_interactivo.html"
    fig.write_html(out)
    print(f"Gráfico HTML guardado en: {out}")

Overwriting src/r1_visualization.py


Dibuja un mapa en 2D (con PCA y t-SNE) y etiqueta cada grupo automáticamente leyendo sus términos más frecuentes. Esto permite explorar visualmente si se ha conseguido la deseada homogeneidad intra-cluster.

# **Bloque 5**

In [ ]:
%%writefile src/r1_inference.py
import numpy as np
from sklearn.metrics import pairwise_distances
from sentence_transformers import SentenceTransformer

def inferir_y_buscar_vecinos(nuevo_texto, modelo_clustering, X_train, labels_train, textos_train_raw):
    # Vectorizamos el texto nuevo usando el mismo transformer
    sbert = SentenceTransformer("all-MiniLM-L6-v2")
    vec_nuevo = sbert.encode([nuevo_texto]).astype(np.float32)

    # Predecimos a qué cluster pertenece
    cluster_pred = int(modelo_clustering.predict(vec_nuevo)[0])

    # Filtramos los textos de Train que están en ese mismo clúster
    idx_cluster = np.where(labels_train == cluster_pred)[0]
    X_sub = X_train[idx_cluster]

    # Calculamos la distancia coseno para buscar los 3 más cercanos
    dists = pairwise_distances(vec_nuevo, X_sub, metric="cosine")[0]
    top_3_locales = np.argsort(dists)[:3]

    print(f"\n--- INFERENCIA SOBRE NUEVO TEXTO ---")
    print(f"Texto: '{nuevo_texto[:100]}...'")
    print(f"Pertenece al CLÚSTER {cluster_pred}")
    print("Top 3 vecinos más parecidos:")
    for rank, loc_idx in enumerate(top_3_locales, 1):
        idx_global = idx_cluster[loc_idx]
        print(f" {rank}. (Dist: {dists[loc_idx]:.3f}) -> {textos_train_raw[idx_global][:120]}...")

Overwriting src/r1_inference.py


Coge textos nuevos, predice a qué clúster van y busca sus vecinos. Además, ejecuta un experimento usando el índice interno Silhouette para medir el compromiso matemático entre la cohesión (intra-cluster) y la separabilidad (inter-cluster), demostrando numéricamente si el modelo es estable en Test.

# **Bloque 6**

In [ ]:
# EJECUTAMOS LOS SCRIPTS EN ORDEN
import sys
sys.path.append("src")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from r1_data_prep import run_preprocessing_and_eda
from r1_vectorizers import generate_all_representations
from r1_visualization import plot_interactive
from r1_inference import inferir_y_buscar_vecinos

# 1. Limpieza
df_tr, df_te = run_preprocessing_and_eda()

# 2. Vectores
generate_all_representations()

# 3. Hacemos un clustering de prueba (KMeans con 5 clusters) usando los vectores de SBERT
X_sbert = np.load("data/embeddings/sbert_train.npy")
kmeans = KMeans(n_clusters=5, random_state=42).fit(X_sbert)
labels = kmeans.labels_

# 4. Generamos el gráfico interactivo
plot_interactive(X_sbert, labels, df_tr['raw_text'].tolist(), df_tr['clean_text'].tolist())

# 5. Probamos con un texto de Test (inferencia)
texto_prueba = df_te['raw_text'].iloc[0]
inferir_y_buscar_vecinos(texto_prueba, kmeans, X_sbert, labels, df_tr['raw_text'].tolist())

Conectando a PubMed (formato seguro) para extraer 150 abstracts...


README.md:   0%|          | 0.00/3.80k [00:00<?, ?B/s]


--- RESUMEN ---
Descarga y limpieza exitosa. Train: 117 | Test: 30
[BOW] guardado. Forma Train: (117, 2500)
[TFIDF] guardado. Forma Train: (117, 2500)
[LSA] guardado. Forma Train: (117, 100)
Calculando SBERT (Contextual)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

Map:   0%|          | 0/117 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

[EMOTION] guardado. Forma Train: (117, 7)
Gráfico HTML guardado en: results/plots/sbert_tsne_interactivo.html


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


--- INFERENCIA SOBRE NUEVO TEXTO ---
Texto: 'established in 2003 , erste foundation has a history stretching back 190 years to the year 1819 when...'
Pertenece al CLÚSTER 0
Top 3 vecinos más parecidos:
 1. (Dist: 0.819) -> we began our investigation of what high reliability might mean for health care by analyzing what is known about how high...
 2. (Dist: 0.829) -> health is not only related to the absence of the disease , therefore we need to conceptualize and operationalize what he...
 3. (Dist: 0.856) -> care coordination is an important aspect of nursing care especially for elderly patients admitted to an acute care setti...


Conecta todo el flujo en una ejecución rápida usando el algoritmo de clustering particional k-medias como base. Demuestra por pantalla que las instancias se agrupan correctamente y no hay errores de código.

# **Bloque 7**

In [ ]:
# 1. Congelar librerías
!pip freeze | grep -E "numpy|pandas|scikit-learn|nltk|gensim|sentence-transformers|pysentimiento|plotly" > requirements.txt

# 2. Escribir README.md legal
readme = """# Agrupamiento No Supervisado - Literatura Biomédica (MeDAL)
*Dataset original:* Wen, Z., Lu, X. H., & Reddy, S. (2020). MeDAL.
*Aviso legal:* Courtesy of the U.S. National Library of Medicine.

## Ejecución Rápida
1. Instalar dependencias: `pip install -r requirements.txt`
2. Ejecutar preparación: `python src/r1_data_prep.py`
3. Generar vectores: `python src/r1_vectorizers.py`
"""
with open("README.md", "w") as f: f.write(readme)

# 3. Comprimir
!tar --exclude='__pycache__' --exclude='.config' --exclude='sample_data' -czvf entrega_practica_r1.tgz README.md requirements.txt src/ data/ results/

# 4. Descargar archivo automáticamente (si usas Chrome/Firefox)
from google.colab import files
files.download("entrega_practica_r1.tgz")

README.md
requirements.txt
src/
src/r1_inference.py
src/r1_data_prep.py
src/r1_visualization.py
src/r1_vectorizers.py
data/
data/raw/
data/raw/medal_train_sample.csv
data/embeddings/
data/embeddings/sbert_train.npy
data/embeddings/lsa_train.npy
data/embeddings/lsa_test.npy
data/embeddings/emotion_test.npy
data/embeddings/bow_test.npy
data/embeddings/models/
data/embeddings/models/tfidf.pkl
data/embeddings/models/bow.pkl
data/embeddings/emotion_train.npy
data/embeddings/tfidf_test.npy
data/embeddings/bow_train.npy
data/embeddings/sbert_test.npy
data/embeddings/tfidf_train.npy
data/processed/
data/processed/train_clean.csv
data/processed/test_clean.csv
results/
results/tables/
results/plots/
results/plots/sbert_tsne_interactivo.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Escribe el manual (README.md) anotando las versiones exactas, y comprime todo el sistema en un archivo .tgz.